# CytoBridge: ARISTA salamander brain regeneration

This notebook loads an aligned AnnData file and a trained model checkpoint, reads the
`arista` settings from the installed CytoBridge package, and runs interpolation and
downstream calculations through package APIs.

Required inputs are an aligned `.h5ad` file and a compatible checkpoint directory. Install
`CytoBridge[all]` in the current Jupyter kernel. A CUDA device is recommended when using the
full interpolation grid. Files created by the workflow are written below
`tutorial_outputs/arista` by default.


In [ ]:
from __future__ import annotations

from importlib import resources
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import yaml

import CytoBridge as cb
from CytoBridge.workflow import (
    WorkflowOptions,
    build_workflow_plan,
    load_workflow_config,
    render_workflow_plan,
    run_workflow,
)

DATASET_PRESET = 'arista'
workflow_preset, workflow_preset_source = load_workflow_config(DATASET_PRESET)
dataset_preset = workflow_preset["dataset"]
scientific_preset = workflow_preset["scientific"]
training_preset = workflow_preset["train"]
downstream_preset = workflow_preset["downstream"]

SEED = int(scientific_preset["seed"])
ALPHA_EXPRESS = float(scientific_preset["alpha_express"])
ALPHA_SPATIAL = float(scientific_preset["alpha_spatial"])
K_NEIGHBORS = int(scientific_preset["classifier_k"])
INTERACTION_CUTOFF = float(training_preset["interaction_cutoff"])
EDGE_PREDICTOR_THRESHOLD = float(training_preset["edge_predictor_threshold"])
CLASSIFIER_EPOCHS = int(downstream_preset["classifier_epochs"])
CLASSIFIER_HIDDEN_SIZE = int(downstream_preset["classifier_hidden_size"])
CLASSIFIER_LR = float(downstream_preset["classifier_lr"])
CLASSIFIER_BEST_METRIC = str(downstream_preset["classifier_best_metric"])
CLASSIFIER_STRICT_STRATIFICATION = bool(
    downstream_preset["classifier_strict_stratification"]
)
PRESET_OBSERVED_TIMES = [float(t) for t in downstream_preset["observed"]]
PRESET_INTERPOLATED_TIMES = [
    float(t) for t in downstream_preset["interpolated"]
]
PRESET_SDE_N_SAMPLES = downstream_preset["sde_n_samples"]
SDE_DT = float(downstream_preset["sde_dt"])
SPLIT_SDE_DT = float(downstream_preset["split_sde_dt"])
SPLIT_SIGMA = float(downstream_preset["split_sigma"])
SPLIT_GROWTH_ALPHA = float(downstream_preset["split_growth_alpha"])
LINEAGE_ENABLED = bool(downstream_preset.get("lineage_enabled", False))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# False uses midpoint interpolation and caps generated particles at 5,000.
# True uses the interpolation grid and particle count from the installed preset.
# Solver settings are read from the same preset in both modes.
RUN_FULL_SCOPE = False
COMPACT_PARTICLES = 5_000

## Optional raw-data preprocessing and training

The paper figures can be reproduced from the aligned H5AD and matching checkpoint used
below. To start from a raw H5AD instead, set `RUN_FROM_RAW=True`. The packaged dataset
preset then runs expression preprocessing, spatial alignment, interaction-graph
construction, edge-predictor fitting, and model training as one matched workflow. This is
disabled by default because it is the full training path.

In [ ]:
RAW_H5AD = Path(f"inputs/{DATASET_PRESET}_raw.h5ad")
RAW_WORKFLOW_DIR = Path(f"tutorial_outputs/{DATASET_PRESET}_from_raw")
RUN_FROM_RAW = False

raw_workflow_result = None
if RUN_FROM_RAW:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Raw input not found: {RAW_H5AD}")
    if RAW_WORKFLOW_DIR.exists():
        raise FileExistsError(
            f"Choose a new raw-workflow output directory: {RAW_WORKFLOW_DIR}"
        )

    raw_options = WorkflowOptions(
        input_h5ad=RAW_H5AD,
        output_dir=RAW_WORKFLOW_DIR,
        device=DEVICE,
        steps=("preprocess", "train"),
        train=True,
    )
    raw_plan = build_workflow_plan(
        workflow_preset,
        source=workflow_preset_source,
        options=raw_options,
    )
    print(render_workflow_plan(raw_plan))
    raw_workflow_result = run_workflow(workflow_preset, options=raw_options)
else:
    print("Raw preprocessing and training are disabled; using aligned inputs below.")

## 1. Set input and output paths

Set `ALIGNED_H5AD`, `MODEL_DIR`, and `OUTPUT_DIR` in the next cell. With
`LR_DATABASE_OVERRIDE=None`, the notebook uses the LR table packaged for `arista`.
`EDGE_PREDICTOR_PATH` is required only when `RUN_TRAINING=True`.


To prepare the aligned input from raw data, the `arista` preset reads the complete
16,379-gene `Regeneration.h5ad`, uses `layers['counts']` once, selects highly variable genes
by batch, fits a pooled PCA basis across eight batches, retains LR subunits, and aligns the
2, 5, 10, 15, and 20 DPI batches. The resulting aligned input contains 46,209 cells.


In [ ]:
ALIGNED_H5AD = Path("inputs/arista_aligned.h5ad")
MODEL_DIR = Path("inputs/arista_model")
if raw_workflow_result is not None:
    ALIGNED_H5AD = Path(raw_workflow_result["outputs"]["aligned_h5ad"])
    MODEL_DIR = Path(raw_workflow_result["outputs"]["model_dir"])
OUTPUT_DIR = Path("tutorial_outputs/arista")

# Use the database packaged for this dataset unless an override is supplied.
LR_DATABASE_OVERRIDE: Path | None = None
if LR_DATABASE_OVERRIDE is None:
    LR_DATABASE = cb.pp.bundled_graph_database_path(DATASET_PRESET)
else:
    LR_DATABASE = Path(LR_DATABASE_OVERRIDE).expanduser().resolve()
    if not LR_DATABASE.is_file():
        raise FileNotFoundError(f"Custom LR database not found: {LR_DATABASE}")

# A new training run needs the dataset-specific edge-classifier checkpoint.
# Existing trained models do not need this path here.
EDGE_PREDICTOR_PATH: Path | None = None
RUN_TRAINING = False

required_external_inputs = {
    "aligned_h5ad": ALIGNED_H5AD,
    **(
        {"edge_predictor_path": EDGE_PREDICTOR_PATH}
        if RUN_TRAINING
        else {"model_dir": MODEL_DIR}
    ),
}
missing_inputs = [
    name
    for name, path in required_external_inputs.items()
    if path is None or not Path(path).exists()
]
if missing_inputs:
    missing_lines = "\n".join(
        f"  - {name}: {required_external_inputs[name]}" for name in missing_inputs
    )
    raise FileNotFoundError(
        f"Provide the dataset/checkpoint files for '{DATASET_PRESET}'.\n"
        f"Missing required input(s):\n{missing_lines}\n"
        "RUN_TRAINING=False expects an existing model directory; set it to True only "
        "for a deliberate new fit and provide EDGE_PREDICTOR_PATH."
    )

pd.Series(
    {
        "workflow_preset": workflow_preset_source,
        "aligned_h5ad": ALIGNED_H5AD,
        "model_dir": MODEL_DIR,
        "lr_database": LR_DATABASE,
        "lr_database_source": (
            "wheel_bundled" if LR_DATABASE_OVERRIDE is None else "explicit_override"
        ),
        "device": DEVICE,
        "run_training": RUN_TRAINING,
    }
)

### Time and expression fields


`obs['time_point_processed']` is the numeric model axis. The raw-data adapter reads biological
stage from `obs['Batch']`, constructs cell identifiers from `Batch` and `CellID`, and writes
model times 0–4. Keep biological stage labels in a separate column for plotting.

The next cell reads the observation and coordinate keys from the dataset preset and checks
that the aligned input contains every required field and observed time point.


In [ ]:
adata = sc.read_h5ad(ALIGNED_H5AD)
TIME_KEY = str(dataset_preset["time_key"])
ANNOTATION_KEY = str(dataset_preset["annotation_key"])
LATENT_KEY = str(dataset_preset["obsm_key"])
SPATIAL_KEY = str(dataset_preset["spatial_key"])
CONCAT_SPATIAL = bool(dataset_preset.get("concat_spatial", True))

required_obs = {TIME_KEY, ANNOTATION_KEY}
required_obsm = {LATENT_KEY, SPATIAL_KEY}
assert required_obs.issubset(adata.obs.columns)
assert required_obsm.issubset(adata.obsm)

available_observed_times = sorted(
    pd.to_numeric(adata.obs[TIME_KEY], errors="raise").unique().astype(float)
)
missing_preset_times = [
    time
    for time in PRESET_OBSERVED_TIMES
    if not any(np.isclose(time, value) for value in available_observed_times)
]
if missing_preset_times:
    raise ValueError(f"Aligned input is missing preset time anchors: {missing_preset_times}")
observed_times = list(PRESET_OBSERVED_TIMES)
input_summary = pd.Series(
    {
        "cells": adata.n_obs,
        "genes": adata.n_vars,
        "available_model_times": available_observed_times,
        "preset_observed_times": observed_times,
        "spatial_dimensions": adata.obsm[SPATIAL_KEY].shape[1],
        "latent_dimensions": adata.obsm[LATENT_KEY].shape[1],
        "cell_types": adata.obs[ANNOTATION_KEY].astype(str).nunique(),
    }
)
input_summary


## 2. Load the packaged preset

`load_workflow_config` supplies the dataset schema, numerical settings, graph cutoff, edge
predictor threshold, and downstream parameters. The training YAML is opened from the
installed package with `importlib.resources` and passed to the training or loading API.


In [ ]:
training_config_name = str(training_preset["config"])
training_config_resource = (
    resources.files("CytoBridge")
    .joinpath("configs")
    .joinpath(training_config_name)
)
if not training_config_resource.is_file():
    raise FileNotFoundError(
        f"Installed CytoBridge wheel is missing {training_config_name!r}."
    )

config = yaml.safe_load(training_config_resource.read_text(encoding="utf-8"))
config["seed"] = SEED
config["ckpt_dir"] = str(MODEL_DIR)
config["training"]["defaults"]["alpha_express"] = ALPHA_EXPRESS
config["training"]["defaults"]["alpha_spatial"] = ALPHA_SPATIAL

interaction_config = config["model"]["interaction_net"]
interaction_config["cutoff"] = INTERACTION_CUTOFF
interaction_config["edge_predictor_thre"] = EDGE_PREDICTOR_THRESHOLD
interaction_config["edge_predictor_path"] = (
    None if EDGE_PREDICTOR_PATH is None else str(EDGE_PREDICTOR_PATH)
)

resolved_settings = pd.Series(
    {
        "workflow_preset": workflow_preset_source,
        "training_config_resource": training_config_name,
        "alpha_express": ALPHA_EXPRESS,
        "alpha_spatial": ALPHA_SPATIAL,
        "seed": SEED,
        "classifier_k": K_NEIGHBORS,
        "interaction_cutoff": INTERACTION_CUTOFF,
        "edge_predictor_threshold": EDGE_PREDICTOR_THRESHOLD,
        "edge_predictor_path_for_new_training": EDGE_PREDICTOR_PATH,
        "preset_scope_enabled": RUN_FULL_SCOPE,
        "preset_observed_times": PRESET_OBSERVED_TIMES,
        "preset_interpolated_times": PRESET_INTERPOLATED_TIMES,
        "preset_sde_n_samples": PRESET_SDE_N_SAMPLES,
        "sde_dt": SDE_DT,
        "split_sde_dt": SPLIT_SDE_DT,
        "split_sigma": SPLIT_SIGMA,
        "split_growth_alpha": SPLIT_GROWTH_ALPHA,
    }
)
resolved_settings


## 3. Optional training

`RUN_TRAINING` is `False` by default. When enabled, `cb.tl.fit` receives the loaded AnnData,
packaged training configuration, device, checkpoint directory, interaction cutoff, and edge
predictor settings. Leave it disabled to use the checkpoint already present in `MODEL_DIR`.


In [ ]:
if RUN_TRAINING:
    cb.tl.fit(
        adata,
        config=config,
        device=DEVICE,
        ckpt_dir=MODEL_DIR,
        interaction_cutoff=INTERACTION_CUTOFF,
        edge_predictor_path=str(EDGE_PREDICTOR_PATH),
        edge_predictor_threshold=EDGE_PREDICTOR_THRESHOLD,
        evaluate_after_training=False,
    )
else:
    print("Training skipped. Set RUN_TRAINING=True only for a deliberate new run.")


## 4. Load the model checkpoint

The model dimension is derived from the aligned spatial coordinates and expression PCs.
`load_dynamical_model_from_dir` loads the configured dynamical and score stages and returns
the model together with its runtime components.


In [ ]:
spatial_dim = int(adata.obsm[SPATIAL_KEY].shape[1])
latent_dim = int(adata.obsm[LATENT_KEY].shape[1])
model_dim = spatial_dim + latent_dim if CONCAT_SPATIAL else latent_dim

loaded = cb.tl.load_dynamical_model_from_dir(
    MODEL_DIR,
    dim=model_dim,
    device=DEVICE,
)
runtime = cb.tl.build_dynamical_runtime(loaded)

pd.Series(
    {
        "model_dimension": model_dim,
        "dynamical_stage": loaded.weight_stage,
        "score_stage": loaded.score_stage,
    }
)


## 5. Calculate interpolated states and cell labels

With `RUN_FULL_SCOPE=False`, the notebook uses one midpoint per observed interval and caps
the generated population for a shorter run. Setting it to `True` uses the interpolation grid
and sample count stored in the preset. Classifier and SDE parameters are read from the same
preset. The classifier neighbor count is read from the preset (`k=10`).

Spatial warping is disabled. Numerical downstream calls use `communication_adata_dict`,
which contains the unwarped states.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
aligned_table, _ = cb.tl.adata_to_aligned_dataframe(
    adata,
    time_key=TIME_KEY,
    obsm_key=LATENT_KEY,
    spatial_key=SPATIAL_KEY,
    concat_spatial=CONCAT_SPATIAL,
    annotation_key=ANNOTATION_KEY,
)

compact_interpolated_times = [
    (left + right) / 2.0
    for left, right in zip(observed_times[:-1], observed_times[1:])
]
if RUN_FULL_SCOPE:
    interpolation_times = list(PRESET_INTERPOLATED_TIMES)
    analysis_particles = (
        None if PRESET_SDE_N_SAMPLES is None else int(PRESET_SDE_N_SAMPLES)
    )
    scope_label = "packaged preset"
else:
    interpolation_times = compact_interpolated_times
    analysis_particles = COMPACT_PARTICLES
    if PRESET_SDE_N_SAMPLES is not None:
        analysis_particles = min(analysis_particles, int(PRESET_SDE_N_SAMPLES))
    scope_label = "compact walkthrough"
analysis_times = sorted(set(observed_times + interpolation_times))
time_values = pd.to_numeric(adata.obs[TIME_KEY], errors="raise").to_numpy(float)
initial_population_size = int(np.isclose(time_values, observed_times[0]).sum())
evaluation_particles = (
    initial_population_size
    if analysis_particles is None
    else min(initial_population_size, int(analysis_particles))
)

pd.Series({
    "scope": scope_label,
    "analysis_times": analysis_times,
    "particle_cap": analysis_particles,
    "split_sde_dt": SPLIT_SDE_DT,
    "split_sigma": SPLIT_SIGMA,
    "split_growth_alpha": SPLIT_GROWTH_ALPHA,
})

trajectory = cb.tl.run_interpolation_workflow(
    df=aligned_table,
    dim=model_dim,
    annotation_key=ANNOTATION_KEY,
    runtime=runtime,
    device=DEVICE,
    output_dir=str(OUTPUT_DIR / "interpolation"),
    requested_plot_points=analysis_times,
    interp_time_points=interpolation_times,
    max_observed_timepoints=len(observed_times),
    use_real_for_observed=True,
    classifier_cache_path=str(OUTPUT_DIR / "classifier_resmlp.pt"),
    classifier_adata=adata,
    classifier_time_key=TIME_KEY,
    classifier_obsm_key=LATENT_KEY,
    classifier_spatial_key=SPATIAL_KEY,
    classifier_concat_spatial=CONCAT_SPATIAL,
    classifier_epochs=CLASSIFIER_EPOCHS,
    classifier_hidden_size=CLASSIFIER_HIDDEN_SIZE,
    classifier_lr=CLASSIFIER_LR,
    classifier_test_size=0.1,
    classifier_best_metric=CLASSIFIER_BEST_METRIC,
    classifier_strict_stratification=CLASSIFIER_STRICT_STRATIFICATION,
    classifier_knn_neighbors=K_NEIGHBORS,
    sde_n_samples=analysis_particles,
    skip_nonsplit_sde=not LINEAGE_ENABLED,
    sde_dt=SDE_DT,
    split_sde_dt=SPLIT_SDE_DT,
    split_sigma_scalar=SPLIT_SIGMA,
    split_growth_alpha=SPLIT_GROWTH_ALPHA,
    split_interaction_m=1024,
    spatial_warp_to_observed=False,
    random_seed=SEED,
)

pd.Series(
    {
        "time_grid": trajectory.ts_points,
        "interpolated_times": trajectory.interp_points,
        "classifier_accuracy": trajectory.classifier_accuracy,
        "classifier_balanced_accuracy": trajectory.classifier_balanced_accuracy,
        "knn_neighbors": K_NEIGHBORS,
        "scope": scope_label,
    }
)


## 6. Calculate cell-type composition

`summarize_label_composition` converts the assigned labels at each requested time into a
table of counts and fractions stored in `composition`.


In [ ]:
labels_by_time = [
    trajectory.adata_dict[key].obs[ANNOTATION_KEY].astype(str).to_numpy()
    for key in trajectory.time_keys
]
composition = cb.tl.summarize_label_composition(
    labels_by_time,
    trajectory.ts_points,
)
composition.head(10)


## 7. Calculate velocity and growth

`compute_velocity_components_from_adata` recalculates drift, interaction, score, and full
velocity for each observed time slice. `reuse_if_present=False` ensures that the arrays are
computed from the loaded checkpoint. `evaluate_growth_by_timepoint` evaluates the generated,
unwarped states.


In [ ]:
velocity = cb.tl.compute_velocity_components_from_adata(
    adata,
    loaded.model,
    dim=model_dim,
    interaction_m=1024,
    interaction_threshold=INTERACTION_CUTOFF,
    device=DEVICE,
    time_key=TIME_KEY,
    obsm_key=LATENT_KEY,
    spatial_key=SPATIAL_KEY,
    concat_spatial=CONCAT_SPATIAL,
    write_to_adata=True,
    reuse_if_present=False,
)

velocity_summary = pd.DataFrame(
    {
        name: {
            "mean_norm": np.linalg.norm(values, axis=1).mean(),
            "median_norm": np.median(np.linalg.norm(values, axis=1)),
        }
        for name, values in velocity.items()
        if name in {"drift", "interaction", "score", "full"}
    }
).T
velocity_summary


In [ ]:
growth = cb.tl.evaluate_growth_by_timepoint(
    trajectory.communication_adata_dict,
    loaded.model,
    time_points=trajectory.ts_points,
    time_keys=trajectory.time_keys,
    annotation_key=ANNOTATION_KEY,
    spatial_key="spatial",
    device=DEVICE,
)
growth.groupby("time")["growth_rate"].agg(["mean", "median", "std"])


## 8. Calculate sparse cell-type communication

`compute_timepoint_communications` builds spatial candidate edges, applies the configured
edge predictor, and aggregates attention by cell type. Candidate edges are processed in
batches; `save_dense_attention_matrix=False` leaves the dense cell-by-cell matrix disabled.
Each returned record includes selection counts in `record['edge_selection']`, and files are
written below `OUTPUT_DIR / 'communication'`.


In [ ]:
communications = cb.tl.compute_timepoint_communications(
    adata_dict=trajectory.communication_adata_dict,
    time_points=trajectory.ts_points,
    annotation_key=ANNOTATION_KEY,
    f_net=runtime.f_net,
    device=DEVICE,
    out_dir=str(OUTPUT_DIR / "communication"),
    save_dense_attention_matrix=False,
    max_cells_per_timepoint=analysis_particles,
    random_seed=SEED,
)
{time: record["edge_selection"] for time, record in communications.items()}


## 9. Calculate ligand–receptor trajectories

`project_communication_to_lr_timecourses` reconstructs per-cell log1p expression from the
PCA state and combines it with the communication matrices. The call requires every complex
subunit and uses the minimum expressed subunit (`complex_mode='min'`). It returns pair-level
time courses and coverage tables in `lr_projection`.


In [ ]:
lr_projection = cb.tl.project_communication_to_lr_timecourses(
    trajectory.communication_adata_dict,
    reference_adata=adata,
    communications=communications,
    lr_database=LR_DATABASE,
    time_points=trajectory.ts_points,
    annotation_key=ANNOTATION_KEY,
    matrix_key="M_per_source",
    spatial_dim=spatial_dim,
    expression_space="log1p",
    complex_mode="min",
    require_all_subunits=True,
    observed_adata=adata,
    observed_time_key=TIME_KEY,
    observed_time_points=observed_times,
    observed_annotation_key=ANNOTATION_KEY,
    observed_expression_space="log1p",
    return_type_matrices=False,
)
lr_projection.pair_timecourse.head(10)


## 10. Calculate distribution metrics

`evaluate_model_distributions` starts from the earliest observed population and calculates
W1, W2, and total-mass variation in joint, spatial, and PCA spaces. The call uses the native
model coordinates and returns its tables in `distribution_evaluation`.


In [ ]:
distribution_evaluation = cb.tl.evaluate_model_distributions(
    adata,
    loaded.model,
    time_points=observed_times,
    n_samples=evaluation_particles,
    dt=SDE_DT,
    sigma=SPLIT_SIGMA,
    include_score=True,
    interaction_m=1024,
    max_ot_points=1024,
    structure_max_points=5_000,
    device=DEVICE,
    time_key=TIME_KEY,
    obsm_key=LATENT_KEY,
    spatial_key=SPATIAL_KEY,
    concat_spatial=CONCAT_SPATIAL,
    random_seed=SEED,
    include_initial_time=False,
    verbose=True,
)
distribution_evaluation.metrics.groupby("space")[["w1", "w2", "tmv"]].mean()


## Outputs

The main in-memory results are `trajectory`, `composition`, `velocity_summary`, `growth`,
`communications`, `lr_projection`, and `distribution_evaluation`. Interpolation files, the
classifier cache, and communication files are written below `OUTPUT_DIR`.
